# 🧪 Evaluate Models in `results/`

这个 Notebook 用于评估/可视化你在 `results/` 下训练出的模型：
- **手动指定模型路径**：直接填 `MODEL_PATH`（绝对/相对路径都可）。
- **按算法自动加载最新模型**：只选 `ALGO_TAG`，脚本会自动寻找该算法最新的 `best_model*.pt` 或 `final_actor.pt`。

自动搜索会兼容三种位置：
1) 新结构：`results/<algo>/**/models/`
2) 旧结构：`results/**/models/`（会跳过 `results/eval/`）
3) 更旧：项目根目录下的同名模型文件

评估产物（指标 JSON、图像）**默认不保存**；如需落盘，可在第 3 步把 `SAVE_OUTPUTS=True`，保存到：`results/eval/<algo>/YYYYMMDD/HHMMSS/`（与训练产物分开）。

本 Notebook 面向 **v2 观测/权重**，统一用 `ActorOnlyPolicy.from_checkpoint` 评测纯 Actor；若发现 v1 旧权重会给出提示并指向 `evaluate_scaling.py --evaluate --worker-version v1`。


In [ ]:
import os
import json
from pathlib import Path
from datetime import datetime

import matplotlib.pyplot as plt
import numpy as np
import torch

# 项目模块
from mas_cc.config import (
    DEVICE, SEED, set_seed,
    NUM_FOLLOWERS, NUM_PINNED, MAX_STEPS,
    LEADER_AMPLITUDE, LEADER_OMEGA, LEADER_PHASE,
    LEADER_TRAJECTORY_TYPES, TOP_K, STATE_DIM,
)
from mas_cc.topology import CommunicationTopology
from mas_cc.environment import ModelFreeEnv
from mas_cc.agent import ActorOnlyPolicy
from mas_cc.utils import evaluate_agent, plot_evaluation, collect_trajectory

print('DEVICE:', DEVICE)
print('torch:', torch.__version__)
print(f'v2 obs: STATE_DIM={STATE_DIM} = 10 + 7 × K({TOP_K})')


In [ ]:
def _algo_tag_normalize(tag: str) -> str:
    tag = (tag or '').strip().lower()
    if tag in ('masac', 'sac', 'ctdesac', 'ctde_sac'):
        return 'masac'
    if tag in ('mappo', 'ppo', 'ctdemappo', 'ctde_mappo'):
        return 'mappo'
    return tag or 'masac'


# v2 完整 checkpoint 与纯 Actor 产物都在搜索范围内
MODEL_PATTERNS = ('best_model*.pt', 'final_actor.pt')


def find_latest_model_in_results(algo_tag, results_root='results', pattern=MODEL_PATTERNS):
    """按修改时间找最新 v2 模型（完整 checkpoint 或纯 Actor）。"""
    algo_tag = _algo_tag_normalize(algo_tag)
    results_root = Path(results_root)
    patterns = (pattern,) if isinstance(pattern, str) else tuple(pattern)

    candidates = [p for pat in patterns for p in results_root.rglob(pat)
                  if p.is_file() and p.parent.name == 'models'
                  and 'eval' not in p.relative_to(results_root).parts]
    legacy_files = [p for pat in patterns for p in Path('.').glob(pat) if p.is_file()]
    for paths in (candidates, legacy_files):
        for path in sorted(paths, key=lambda p: p.stat().st_mtime_ns, reverse=True):
            if guess_algo_from_model_path(path) == algo_tag:
                return str(path)
    raise FileNotFoundError(
        f"未找到 v2 模型：algo={algo_tag}, pattern={patterns}。\n"
        f"已搜索：{results_root}（跳过 eval）和项目根目录。"
    )


def inspect_checkpoint(path):
    """识别 v2 完整 checkpoint / 纯 Actor / v1 旧权重，返回 (kind, detail)。"""
    try:
        ckpt = torch.load(path, map_location='cpu', weights_only=True)
    except Exception as exc:
        return 'v1', f'无法按 v2 格式读取（{type(exc).__name__}: {exc}）'
    if not isinstance(ckpt, dict):
        return 'v1', '权重不是 checkpoint dict'
    weights = ckpt.get('actor', ckpt)
    if not isinstance(weights, dict):
        return 'v1', '未找到 actor state_dict'
    if any('pos_embed' in str(k) for k in weights):
        return 'v1', 'Actor 含 pos_embed（旧位置编码）'
    version = weights.get('_obs_version')
    if not (torch.is_tensor(version) and version.ndim == 0 and int(version.item()) == 2):
        return 'v1', '缺少 v2 _obs_version 标记'
    tk = weights.get('_top_k')
    if not (torch.is_tensor(tk) and tk.ndim == 0 and int(tk.item()) == TOP_K):
        return 'v1', f'_top_k 与当前 TOP_K={TOP_K} 不匹配'
    training_keys = {'value', 'q1', 'q2', 'q1_target', 'q2_target'}
    kind = 'v2-full' if any(k in ckpt for k in training_keys) else 'v2-actor-only'
    return kind, 'v2 checkpoint，可用 ActorOnlyPolicy.from_checkpoint 直接加载'


def guess_algo_from_model_path(model_path: str) -> str:
    path = Path(model_path).expanduser()
    try:
        # 只读取算法标记，搜索时无需将候选权重分配到 CPU/GPU。
        ckpt = torch.load(path, map_location='meta', weights_only=True)
    except Exception:
        ckpt = None  # 旧格式仍按路径归类，由 inspect_checkpoint 给出加载提示。
    if isinstance(ckpt, dict) and isinstance(ckpt.get('algorithm'), str):
        return _algo_tag_normalize(ckpt['algorithm'])
    for part in (path.stem, *reversed(path.parent.parts)):
        name = part.lower()
        tokens = name.replace('-', '_').split('_')
        if 'mappo' in name or 'ppo' in tokens:
            return 'mappo'
        if 'masac' in name or 'sac' in tokens:
            return 'masac'
    return 'masac'  # 兼容无算法标记的旧 SAC 命名。


def make_eval_output_dirs(algo_tag: str, results_root: str | Path = 'results') -> dict:
    algo_tag = _algo_tag_normalize(algo_tag)
    now = datetime.now()
    date = now.strftime('%Y%m%d')
    t = now.strftime('%H%M%S')

    base = Path(results_root) / 'eval' / algo_tag / date / t
    models_dir = base / 'models'
    figs_dir = base / 'figs'
    base.mkdir(parents=True, exist_ok=True)
    models_dir.mkdir(parents=True, exist_ok=True)
    figs_dir.mkdir(parents=True, exist_ok=True)

    return {
        'base': base,
        'models_dir': models_dir,
        'figs_dir': figs_dir,
    }


## 1) 选择模型

你可以二选一：
- 方式 A：手动指定 `MODEL_PATH`
- 方式 B：只指定 `ALGO_TAG`，自动加载该算法最新模型


In [ ]:
# ========== 方式 A（手动）==========
MODEL_PATH = ''  # e.g. '.../models/best_model_ctde_20f_mappo.pt' 或 '.../models/final_actor.pt'

# ========== 方式 B（自动）==========
ALGO_TAG = 'mappo'  # 'mappo' or 'masac'
AUTO_LOAD_LATEST = True

# 同时识别 v2 完整 checkpoint 与纯 Actor 产物
MODEL_GLOB = ('best_model*.pt', 'final_actor.pt')

# -------- 决策逻辑 --------
if MODEL_PATH:
    model_path = MODEL_PATH
    algo_tag = guess_algo_from_model_path(model_path)
else:
    algo_tag = _algo_tag_normalize(ALGO_TAG)
    if AUTO_LOAD_LATEST:
        model_path = find_latest_model_in_results(algo_tag, results_root='results', pattern=MODEL_GLOB)
    else:
        raise ValueError('AUTO_LOAD_LATEST=False 时请手动填写 MODEL_PATH')

model_path = str(Path(model_path).expanduser())
assert Path(model_path).exists(), f'模型文件不存在: {model_path}'

ckpt_kind, ckpt_detail = inspect_checkpoint(model_path)
print('Selected algo_tag:', algo_tag)
print('Selected model_path:', model_path)
print(f'Checkpoint: {ckpt_kind} — {ckpt_detail}')
if ckpt_kind == 'v1':
    raise RuntimeError(
        '检测到 v1 旧权重（无 v2 _obs_version/_top_k 或含 pos_embed）。\n'
        '本 Notebook 只评测 v2 权重；v1 权重请走冻结场景脚本：\n'
        '  python evaluate_scaling.py --evaluate --worker-version v1 --scenario-manifest scenes.json '
        '--repo-path <v1-repo> --training-seed <seed> --checkpoint old.pt --output v1.json'
    )


## 2) 加载模型并评估（输出数值指标）

默认会评估 `num_eval_episodes` 个 episode，输出平均 reward / 平均跟踪误差 / 平均通信率。


In [ ]:
# 评估参数
num_eval_episodes = 10
seed_for_eval = SEED

# 画图时最多展示多少个跟随者：None=全部；例如 5=只画前 5 个
MAX_PLOT_FOLLOWERS = None

set_seed(seed_for_eval)

# 环境/拓扑（Actor 输入维度固定，与 N 无关）
topology = CommunicationTopology(NUM_FOLLOWERS, num_pinned=NUM_PINNED)
env = ModelFreeEnv(topology)

# v2 评测统一走纯 Actor：无需构造 CTDEMAPPOAgent / CTDESACAgent
policy = ActorOnlyPolicy.from_checkpoint(model_path)

metrics = evaluate_agent(policy, env, num_episodes=num_eval_episodes)
print('Metrics:', metrics)
print('说明：metrics["mean_tracking_error"] 来自环境 info 的跟踪误差；')
print('      plot_evaluation 图中的 avg_error 是位置 MSE（follower_pos 与 leader_pos 的平方误差），')
print('      两者口径不同，请勿混用。')


## 3) （可选）保存评估产物（指标 JSON + 评估图）

默认不保存任何生成物；把下面单元格里的 `SAVE_OUTPUTS=True` 后再运行即可保存到 `results/eval/<algo>/YYYYMMDD/HHMMSS/`。


In [ ]:
# 默认不保存任何生成物；需要落盘时改成 True 并重新运行本单元格
SAVE_OUTPUTS = False

if SAVE_OUTPUTS:
    out = make_eval_output_dirs(algo_tag, results_root='results')
    print('Eval outputs ->', out['base'])

    # 1) 保存指标（同时记录两种口径，避免混淆）
    metrics_path = out['base'] / 'metrics.json'
    payload = {
        'algo': algo_tag,
        'model_path': model_path,
        'checkpoint_kind': ckpt_kind,
        'seed': seed_for_eval,
        'num_eval_episodes': num_eval_episodes,
        'metrics': metrics,
        'metric_note': 'mean_tracking_error=环境 info 口径；avg_error=绘图位置 MSE，二者不同',
        'timestamp': datetime.now().isoformat(timespec='seconds'),
    }
    metrics_path.write_text(json.dumps(payload, indent=2, ensure_ascii=False), encoding='utf-8')
    print('Saved metrics ->', metrics_path)

    # 2) 保存评估图
    fig_path = out['figs_dir'] / f'final_evaluation_{algo_tag}.png'
    plot_evaluation(policy, topology, num_tests=3, save_path=str(fig_path), max_plot_followers=MAX_PLOT_FOLLOWERS)
    print('Saved figure ->', fig_path)
else:
    print('SAVE_OUTPUTS=False：本次仅评估并打印 metrics，不保存 metrics.json/图片。')
    # 仍可在 Notebook 里直接看交互图（不落盘）
    plot_evaluation(policy, topology, num_tests=3, save_path=None, max_plot_followers=MAX_PLOT_FOLLOWERS)


## 可选：固定环境随机化（做可复现对比）

你的环境默认 `reset()` 会随机化 leader 参数/初始状态/拓扑。
如果你想在**同一个固定场景**里对比多个模型，可以把随机化关掉。


In [ ]:
# 固定场景：必须在首次 reset 之前安装，保证第 0 步动作即使用指定轨迹。
fixed_scenario = {
    'leader_id': 0,
    'adj_matrix': topology.adj.detach().cpu().numpy(),
    'pinned_ids': topology.pinned_ids,
    'trajectory_type': LEADER_TRAJECTORY_TYPES[0],
    'leader_amplitude': float(LEADER_AMPLITUDE),
    'leader_omega': float(LEADER_OMEGA),
    'leader_phase': float(LEADER_PHASE),
    'initial_positions': np.zeros(NUM_FOLLOWERS),
    'initial_velocities': np.zeros(NUM_FOLLOWERS),
}

env_fixed = ModelFreeEnv(topology, auto_reset=False)
state = env_fixed.reset(scenario=fixed_scenario)  # pre-reset scenario wiring
metrics_fixed = evaluate_agent(policy, env_fixed, num_episodes=5)
print('Fixed-scenario metrics:', metrics_fixed)

# 同一固定场景的位置 MSE（plot_evaluation 的 avg_error 口径）
traj = collect_trajectory(policy, env_fixed)
pos_mse = float(np.mean((traj['follower_pos'] - traj['leader_pos'][:, None]) ** 2))
print(f'Fixed scenario position MSE (avg_error)={pos_mse:.4f}; '
      f'mean_tracking_error={metrics_fixed["mean_tracking_error"]:.4f}')
fig, axes = plt.subplots(1, 2, figsize=(14, 4))
n_show = MAX_PLOT_FOLLOWERS if MAX_PLOT_FOLLOWERS and MAX_PLOT_FOLLOWERS > 0 else None
for ax, kind, label in zip(axes, ('pos', 'vel'), ('Position', 'Velocity')):
    ax.plot(traj['times'], traj[f'leader_{kind}'], 'r-', label='Leader')
    ax.plot(traj['times'], traj[f'follower_{kind}'][:, :n_show], alpha=0.7)
    ax.set(xlabel='Time (s)', ylabel=label, title=f'Fixed scenario: {label}')
    ax.legend()
    ax.grid(alpha=0.3)
fig.tight_layout()
plt.show()
